# GC Gold Breakout Backtest

This notebook implements the approved GC Globex breakout backtest as a production-style research workflow. The core engine lives in `gold_breakout_backtest.py`; the notebook keeps the analysis auditable by exposing each major stage, its diagnostics, and its validation outputs in sequence.

## Clarifications and Approved Decisions

- Backtest window is restricted to sessions starting on `2018-08-08 UTC`.
- The missing plan parameters are configurable and default to `fixed_ticks = 100` and `atr_multiplier_tpsl = 2.0`.
- Session boundaries are derived from the observed one-hour maintenance gap in the restricted sample because the UTC open shifts seasonally.
- Trigger evaluation uses adjusted levels; fills, transaction costs, and P&L stay in unadjusted price space.
- Open positions are rolled across dominant-contract changes, with rollover costs recorded separately and included in total trade costs.
- The hourly trailing-stop model is a structural upper bound on live win rate; every win-rate output should be read with that caveat in mind.


## 0. Configuration

All strategy, cost, risk, and file-path parameters live in one block. No downstream cell hardcodes strategy constants.


In [ ]:
from pathlib import Path

from gold_breakout_backtest import (
    config_frame,
    make_default_config,
)

config = make_default_config(
    csv_path=Path("glbx-mdp3-20100606-20260325.ohlcv-1h.csv"),
    backtest_start_session_date="2018-08-08",
    starting_capital=100_000.0,
    risk_free_rate=0.0,
    min_session_volume=0,
    tp_sl_mode="prior_day_range",
    fixed_ticks=100,
    atr_multiplier_tpsl=2.0,
    k=0.50,
    atr_period=14,
    atr_multiplier=1.5,
    risk_fraction=0.01,
)

config_frame(config)


## 1. Imports and Environment Setup

Only standard research dependencies are used here: `pandas`, `numpy`, `matplotlib`, and the local backtest module.


In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from gold_breakout_backtest import (
    plot_drawdown_curve,
    plot_equity_curve,
    plot_exit_reason_breakdown,
    plot_heatmap_grid,
    plot_monthly_returns_heatmap,
    plot_session_replay,
    plot_sizing_sensitivity_curve,
    plot_trade_pnl_distribution,
    plot_trail_sensitivity_curve,
    prepare_research_data,
    preview_signal_inputs,
    run_backtest,
    run_sensitivity_analysis,
    run_walk_forward_analysis,
    summarize_audit,
    with_atr_period,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
plt.rcParams["figure.dpi"] = 120


## 2. Data Ingestion and Validation

The loader enforces the raw CSV schema, filters out non-hourly rows and spreads, resolves duplicates deterministically, and records exclusions plus anomaly flags for review.


In [ ]:
prepared = prepare_research_data(config)

display(summarize_audit(prepared.audit))
display(prepared.audit["frames"]["excluded_spreads"].head(10))
display(prepared.audit["frames"]["duplicate_resolution"].head(10))
display(prepared.audit["frames"]["low_volume_sessions_all_outrights"].head(10))


## 3. Continuous Series Construction

The dominant contract is selected by session-level volume, roll dates are logged, and a backward Panama adjustment is applied while preserving the unadjusted OHLC columns in parallel.


In [ ]:
display(prepared.session_table.head(10))
display(prepared.audit["frames"]["roll_log"].head(10))
display(prepared.audit["frames"]["incomplete_or_irregular_sessions"].head(10))


## 4. Feature Engineering

This stage creates previous-session breakout levels, previous-session range, true range, and the shifted Wilder ATR used by the strategy.


In [ ]:
feature_prepared = with_atr_period(prepared, config.atr_period)

feature_prepared.continuous_data[
    [
        "ts_event",
        "session_date",
        "dominant_contract",
        "adj_open",
        "adj_high",
        "adj_low",
        "adj_close",
        "prev_session_high",
        "prev_session_low",
        "prev_session_range",
        "true_range",
        "atr",
    ]
].head(15)


## 5. Signal Generation

Signals are prepared from session-open inputs only. The actual trigger sequence remains path-dependent and is handled inside the sequential engine, but this preview makes the daily order setup auditable.


In [ ]:
signal_preview = preview_signal_inputs(feature_prepared, config, rows=15)
signal_preview


## 6. Portfolio and Execution Engine

The engine processes sessions chronologically, updates trailing stops bar by bar, supports concurrent long and short positions, rolls open positions across contract changes, and records a full trade lifecycle.


In [ ]:
backtest_start = perf_counter()
result = run_backtest(feature_prepared, config)
backtest_elapsed = perf_counter() - backtest_start
print(f"Backtest runtime: {backtest_elapsed:.2f} seconds")

display(result.trade_log.head(10))
display(result.roll_events.head(10))
display(result.order_cancellations.head(10))


## 7. Transaction Cost Model

Round-turn costs are applied at the trade level, and approved rollover costs are added for trades that survive dominant-contract changes.


In [ ]:
cost_summary = result.performance_summary[
    [
        "gross_total_pnl",
        "net_total_pnl",
        "total_cost_drag",
        "cost_drag_pct_of_gross",
        "sizing_floor_events",
    ]
]

cost_summary.to_frame("value")


## 8. Backtest Loop Diagnostics

This view focuses on the chronological engine diagnostics: skipped sessions, concurrency, and margin flags.


In [ ]:
display(result.skipped_sessions.head(10))
display(result.margin_flags.head(10))
display(result.diagnostics["session_sizing"].head(10))
display(result.diagnostics["concurrency_log"].head(10))


## 9. Performance Analytics

The summary includes headline return, drawdown, risk-adjusted metrics, trade statistics, direction splits, and annual breakdowns.


In [ ]:
display(result.performance_summary.to_frame("value"))
display(result.benchmark_summary)
display(result.direction_summary)
display(result.annual_summary)
display(result.exit_breakdown)


## 10. Visualization

Plots focus on analytical clarity: realized equity and drawdown against benchmarks, monthly returns, P&L distributions, and exit composition.


In [ ]:
for figure in [
    plot_equity_curve(result),
    plot_drawdown_curve(result),
    plot_monthly_returns_heatmap(result),
    plot_trade_pnl_distribution(result),
    plot_exit_reason_breakdown(result),
]:
    display(figure)
    plt.close(figure)


## 11. Session Replay

The session replay reconstructs one session from the recorded bars, trade log, cancellations, and skip markers so the breakout logic can be audited visually.


In [ ]:
replay_session_id = (
    int(result.trade_log["entry_session_id"].iloc[0])
    if not result.trade_log.empty
    else int(result.session_table["session_id"].iloc[0])
)

figure = plot_session_replay(result, replay_session_id)
display(figure)
plt.close(figure)


## 12. Sensitivity Analysis

The primary grid reports `atr_multiplier × k` heatmaps by `tp_sl_mode`. For `symmetric_fixed` and `atr_based`, the `k` axis is intentionally repeated because the mode does not consume `k`, but the output shape is kept identical for comparison.

Interpretation note:

- A sharp single-cell optimum is evidence of overfitting risk.
- A broad plateau is evidence of robustness.
- The grid now extends below the prior lower bounds for both `atr_multiplier` and `k`; treat any apparent improvement at those tighter settings as exploratory until it survives walk-forward and slippage realism checks.


In [ ]:
sensitivity_start = perf_counter()
sensitivity = run_sensitivity_analysis(feature_prepared, config)
sensitivity_elapsed = perf_counter() - sensitivity_start
print(f"Sensitivity runtime: {sensitivity_elapsed:.2f} seconds")

primary_grid = sensitivity["primary_grid"]
sizing_grid = sensitivity["sizing_grid"]

display(primary_grid.head(12))
display(sizing_grid)

for metric in ["win_rate", "profit_factor", "sharpe_ratio", "max_drawdown_pct"]:
    figure = plot_heatmap_grid(primary_grid, metric=metric, atr_period=config.atr_period)
    display(figure)
    plt.close(figure)

figure = plot_trail_sensitivity_curve(primary_grid, config)
display(figure)
plt.close(figure)

figure = plot_sizing_sensitivity_curve(sizing_grid)
display(figure)
plt.close(figure)


## 13. Walk-Forward OOS

The walk-forward report uses rolling train/test windows and a deliberately smaller optimization grid than the full sensitivity sweep, so the OOS validation remains computationally practical while still testing parameter stability across time.


In [ ]:
walk_forward_start = perf_counter()
walk_forward = run_walk_forward_analysis(feature_prepared, config)
walk_forward_elapsed = perf_counter() - walk_forward_start
print(f"Walk-forward runtime: {walk_forward_elapsed:.2f} seconds")

display(walk_forward.fold_summary)
display(walk_forward.parameter_stability)
display(walk_forward.optimization_results.head(12))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(walk_forward.oos_equity_curve["ts_event"], walk_forward.oos_equity_curve["net_equity"], label="Stitched OOS net equity")
ax.set_title("Walk-Forward Stitched OOS Equity")
ax.set_ylabel("Equity")
ax.grid(True, alpha=0.2)
ax.legend()
display(fig)
plt.close(fig)


## 14. Validation and Unit Tests

The notebook fails fast if the audited invariants break. These are the core temporal-integrity, cost-accounting, and logging checks required by the plan, adapted where necessary to handle the approved rollover mechanics and the approved unadjusted-fill provenance audit.


In [ ]:
display(result.validation_results)
assert (result.validation_results["status"] == "pass").all(), 'One or more validation checks failed.'
